In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t020/task.json'
with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "frontend异常告警" and occurred between 2026-03-16 05:09:23.289044 and 2026-03-16 05:14:02 (UTC+8). The entity that triggered the alert is an APM operation named "frontend::POST /api/cart" with the entity domain "apm". The user is requesting root cause analysis for this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('frontend异常告警', {'start': '2026-03-16T05:09:23.289044+08:00', 'end': '2026-03-16T05:14:02+08:00'}, {'entity_id': '9e69280c47b4c2f35576402946e3b714', 'entity_name': 'frontend::POST /api/cart', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">frontend异常告警<alert_event event_id="5e988330a33aeaa15baa8ff4dcda7984" trans_id="17shfgdjg116pctvgva24kei15" rule_id="d4107ac6-2a37-4167-9059-16156c90f2bf" rule_name="frontend异常告警" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

# Load the topology.json file to understand the service dependency graph
topology_file_path = 'rca100/cases/t020/topology.json'
with open(topology_file_path, 'r') as file:
    topology_data = json.load(file)

# Extract entities and edges
entities = topology_data.get('entities', [])
edges = topology_data.get('edges', [])

# Identify upstream and downstream dependencies for the 'frontend' service
frontend_service_id = next((entity['id'] for entity in entities if entity['name'] == 'frontend'), None)

# Find upstream and downstream services
upstream_services = [edge['src'] for edge in edges if edge['dst'] == frontend_service_id and edge['relation'] == 'calls']
downstream_services = [edge['dst'] for edge in edges if edge['src'] == frontend_service_id and edge['relation'] == 'calls']

# Map service IDs to their names
service_id_to_name = {entity['id']: entity['name'] for entity in entities}
upstream_service_names = [service_id_to_name[sid] for sid in upstream_services if sid in service_id_to_name]
downstream_service_names = [service_id_to_name[sid] for sid in downstream_services if sid in service_id_to_name]

upstream_service_names, downstream_service_names

```
Out[2]:
```
The 'frontend' service has one upstream dependency: 'frontend-proxy'. It has multiple downstream dependencies, including 'checkout', 'product-catalog', 'inventory', 'cart', 'recommendation', and 'ad', with some services appearing multiple times, such as 'checkout', 'cart', 'ad', and 'recommendation'.

The original code execution output of IPython Kernel is also provided below for reference:

(['frontend-proxy'], ['checkout', 'product-catalog', 'checkout', 'inventory', 'cart', 'recommendation', 'ad', 'cart', 'inventory:9090', 'ad', 'product-catalog', 'recommendation'])```
```

In [ ]:
# In[3]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t020/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all available KPIs and their corresponding entity sets
kpi_entity_sets = metrics_data[['metric', 'entity_set']].drop_duplicates()

kpi_entity_sets

```
Out[3]:
```
The `metrics.parquet` file contains 63 unique KPIs across various entity sets. Some examples include `node_memory_usage_rate`, `node_disk_usage_rate`, and `node_ready_status` under the `k8s.node` entity set, as well as `workload`, `error`, and `slow` under the `apm.service.legacy` entity set. Additionally, `error_rate` is available for both `apm.operation` and `apm.instance` entity sets.

The original code execution output of IPython Kernel is also provided below for reference:

metric          entity_set
0      node_memory_usage_rate            k8s.node
1876     node_disk_usage_rate            k8s.node
2144        node_ready_status            k8s.node
4154   node_pod_running_count            k8s.node
6164    node_not_ready_status            k8s.node
...                       ...                 ...
68659                workload  apm.service.legacy
68665                   error  apm.service.legacy
68668                    slow  apm.service.legacy
78706              error_rate       apm.operation
79147              error_rate        apm.instance

[63 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Filter metrics for the relevant KPIs and entity sets
relevant_metrics = metrics_data[
    ((metrics_data['entity_set'] == 'apm.operation') | (metrics_data['entity_set'] == 'apm.service.legacy')) &
    (metrics_data['metric'].isin(['error_rate', 'error', 'slow']))
]

# Ensure 'value' column is numeric for aggregation
relevant_metrics['value'] = pd.to_numeric(relevant_metrics['value'], errors='coerce')

# Calculate global P95 thresholds for each KPI across all components
p95_thresholds = relevant_metrics.groupby('metric')['value'].quantile(0.95).reset_index()
p95_thresholds.columns = ['metric', 'p95_threshold']

p95_thresholds

```
Out[4]:
```
The global P95 thresholds for the relevant KPIs across all components are as follows: 
- `error`: 1789.9
- `error_rate`: 1.0
- `slow`: 35.0 

These thresholds can be used to identify anomalies in the metrics.

The original code execution output of IPython Kernel is also provided below for reference:

metric  p95_threshold
0       error         1789.9
1  error_rate            1.0
2        slow           35.0```
```

In [ ]:
# In[5]:

import pytz
from datetime import datetime

# Define the time window in UTC+8
start_time = datetime.fromisoformat('2026-03-16T05:09:23.289044+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
end_time = datetime.fromisoformat('2026-03-16T05:14:02+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert microseconds to seconds for filtering
start_timestamp = int(start_time.timestamp() * 1_000_000)
end_timestamp = int(end_time.timestamp() * 1_000_000)

# Filter metrics data for the given time window
filtered_metrics = relevant_metrics[
    (relevant_metrics['time'] >= start_timestamp) & (relevant_metrics['time'] <= end_timestamp)
]

# Filter for 'frontend' service and its dependencies
frontend_related_services = ['frontend', 'frontend-proxy', 'checkout', 'product-catalog', 
                             'inventory', 'cart', 'recommendation', 'ad', 'inventory:9090']
frontend_metrics = filtered_metrics[filtered_metrics['entity_name'].isin(frontend_related_services)]

# Merge with P95 thresholds to identify anomalies
anomalies = frontend_metrics.merge(p95_thresholds, on='metric')
anomalies = anomalies[anomalies['value'] > anomalies['p95_threshold']]

anomalies

```
Out[5]:
```
During the specified time window (2026-03-16 05:09:23.289044 to 2026-03-16 05:14:02 UTC+8), the `frontend` service exhibited anomalies in the `slow` KPI. Specifically, the `slow` metric exceeded the global P95 threshold of 35.0 twice, with values of 54.0 and 60.0. This indicates potential performance issues in the `frontend` service.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set entity_id entity_name metric  value metric_set_id service  p95_threshold
51  1773609113000000    apm  apm.service.legacy              frontend   slow   54.0          None    None           35.0
62  1773609143000000    apm  apm.service.legacy              frontend   slow   60.0          None    None           35.0```
```

In [ ]:
# In[6]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t020/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure relevant columns are numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Convert the alert time window to nanoseconds for filtering
start_time_ns = start_timestamp * 1_000
end_time_ns = end_timestamp * 1_000

# Filter traces related to the 'frontend' service during the alert time window
frontend_traces = traces_data[
    (traces_data['serviceName'] == 'frontend') &
    (traces_data['startTime'] >= start_time_ns) &
    (traces_data['endTime'] <= end_time_ns)
]

# Identify traces with errors or anomalies (statusCode != 0 or unusually high duration)
frontend_traces_with_issues = frontend_traces[
    (traces_data['statusCode'] != 0) | 
    (traces_data['duration'] > p95_thresholds.loc[p95_thresholds['metric'] == 'slow', 'p95_threshold'].values[0] * 1_000_000)
]

# Trace the call chain to find the most downstream faulty service
downstream_faulty_services = frontend_traces_with_issues.groupby('serviceName')['duration'].max().reset_index()

frontend_traces_with_issues, downstream_faulty_services

```
Out[6]:
```
The trace analysis for the `frontend` service during the alert time window identified multiple traces with issues. The most downstream faulty service in the call chain is the `frontend` service itself, which exhibited a maximum trace duration of approximately 2.07 seconds, indicating a significant performance issue. Further investigation into the `frontend` service is required to determine the root cause of the anomaly.

The original code execution output of IPython Kernel is also provided below for reference:

(                                 traceId            spanId      parentSpanId kind                                          spanName  ...                                         attributes  __topic__      __tag__:__pack_id__ links events
400059  62d601e93a227903bf0711467f631f66  30d41c3b21f00d09  b8b33c6b95d39503    3                 grpc.oteldemo.CartService/AddItem  ...  {"otel.scope.name":"@opentelemetry/instrumenta...             61213E2A7B8AF035-6D2878  None   None
400060  0e90f7bb6e842a43e50f84e72f4bb42f  4be6bd6df4a08173  fba5f07906f8ef88    3    grpc.oteldemo.ProductCatalogService/GetProduct  ...  {"otel.scope.name":"@opentelemetry/instrumenta...             61213E2A7B8AF035-6D2878  None   None
400061  7be2129e99f037c9edae767ecf77b31b  ed2b6fbe0d17fc83  9398e48ca1fff38b    3                 grpc.oteldemo.CartService/AddItem  ...  {"otel.scope.name":"@opentelemetry/instrumenta...             61213E2A7B8AF035-6D2878  None   None
400062  bb9a5718d22df8602abb0a4ea340ba07  70aec542fdae58c4  3eaed677ef20d647    3          grpc.oteldemo.CheckoutService/PlaceOrder  ...  {"otel.scope.name":"@opentelemetry/instrumenta...             61213E2A7B8AF035-6D2878  None   None
400063  0e31673709f6713e3ab89d21e0cf4f00  3a2857594a16bcfc  98e10dd20e6fcf9b    3    grpc.oteldemo.ProductCatalogService/GetProduct  ...  {"otel.scope.name":"@opentelemetry/instrumenta...             61213E2A7B8AF035-6D2878  None   None
...                                  ...               ...               ...  ...                                               ...  ...                                                ...        ...                      ...   ...    ...
499974  134190dca7561c8cdba40e1a4e5cbb51  af191ae53439b5f5  bc79b51a619a715c    2                          GET /api/recommendations  ...  {"otel.scope.name":"next.js","rpc":"GET /api/r...             12D2135A6D859D83-6F5C73  None   None
499975  6a62188e01fc1804af685d6441e47b87  ea8301ea0d9d3f47  5b0bb73e448ab07c    1  executing api route (pages) /api/recommendations  ...  {"http.status_code":"200","otel.scope.name":"n...             12D2135A6D859D83-6F5C73  None   None
499976  6a62188e01fc1804af685d6441e47b87  5b0bb73e448ab07c  fccf1e301c5032c0    2                          GET /api/recommendations  ...  {"otel.scope.name":"next.js","rpc":"GET /api/r...             12D2135A6D859D83-6F5C73  None   None
499977  4dfd3430bd4e983148d33028a14239ab  cc19c61d622405c2  8db117234f20cf47    1         executing api route (pages) /api/products  ...  {"http.status_code":"200","otel.scope.name":"n...             12D2135A6D859D83-6F5C73  None   None
499978  4dfd3430bd4e983148d33028a14239ab  8db117234f20cf47  7d179e3b612399ad    2                                 GET /api/products  ...  {"otel.scope.name":"next.js","rpc":"GET /api/p...             12D2135A6D859D83-6F5C73  None   None

[34049 rows x 21 columns],   serviceName    duration
0    frontend  2072335380)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t020/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is in numeric format for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs related to the 'frontend' service during the alert time window
frontend_logs = logs_data[
    (logs_data['_container_name_'].str.contains('frontend', case=False, na=False)) &
    (logs_data['_time_'] >= start_timestamp) &
    (logs_data['_time_'] <= end_timestamp)
]

# Search for error messages or relevant information in the logs
frontend_error_logs = frontend_logs[frontend_logs['content'].str.contains('error', case=False, na=False)]

frontend_error_logs

```
Out[7]:
```
No error messages or relevant information were found in the logs for the `frontend` service during the alert time window. This suggests that the root cause of the performance issue may not be directly logged as an error in the `frontend` service. Further investigation into other telemetry data or upstream/downstream services may be necessary.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, _image_name_, __topic__, __tag__:__pack_id__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__]
Index: []

[0 rows x 15 columns]```
```